# PHORA-LI v2 / PHORA+ — AMPLIFAI New-Data Paper Notebook

**Designed for the updated public release:** four `batch_*.zip` archives plus separate `train_metadata.csv` and `val_metadata.csv`.

This notebook deliberately separates **method development** from the **official 59-case validation holdout**:

1. unpack the batch archives once and recursively discover lesion cases;
2. encode the updated metadata safely (`Non-rim APHE` and `Rim APHE` are separate training concepts);
3. extract three inference-safe feature views:
   - spatial/physiological features from `phora_features.py`,
   - optional PyRadiomics features,
   - local liver-reference + optimal-transport features;
4. distill voxel annotation masks into **training-only spatial-burden targets**;
5. compare strong classical baselines, flat boosting, multi-view stacking, simple hierarchy, and **PHORA+ v2**;
6. perform architecture and feature-family ablations on training CV only;
7. evaluate the frozen method once on the official validation split;
8. generate bootstrap CIs, concept-recovery, uncertainty, domain-shift, and failure-mode analyses;
9. after the method is frozen, retrain on train+validation for the final challenge model.

### Main PHORA+ v2 additions

- cross-fitted binary clinical concepts: non-rim APHE, rim APHE, washout, capsule;
- cross-fitted **spatial annotation burden distillation**;
- local lesion-vs-perilesional reference kinetics;
- Wasserstein/Jensen–Shannon phase-distribution distances;
- spatial autocorrelation and hotspot topology;
- direct + factorized special-category gates;
- shared all-threshold ordinal learning;
- metric-aware nested routing;
- **probabilistic LI-RADS constraints**: LR-5 is softly vetoed when predicted non-rim APHE is low, while rim APHE softly supports LR-M.

## 0. Environment

Use the same environment as the previous PHORA notebook. PyRadiomics is optional but strongly recommended for the paper baseline/feature-view experiments.

In [46]:
# Uncomment only if a package is missing.
# %pip install numpy==1.26.4 scipy pandas scikit-learn nibabel xgboost matplotlib joblib tqdm
# %pip install SimpleITK pyradiomics

from pathlib import Path
import os, sys, json, zipfile, shutil, subprocess, warnings, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report,
    roc_auc_score, average_precision_score
)

warnings.filterwarnings('ignore', category=FutureWarning)

## 1. Paths and experiment switches

The defaults below match the folder shown in your screenshot. If the notebook is not stored beside the PHORA code files, edit `PROJECT_DIR`.

In [47]:
NEW_DATA_ROOT = Path(r'E:\HCC_MICCAI26\new_data')
PROJECT_DIR = Path.cwd()                  # folder containing phora_features.py and phora_plus_v2.py
TRAIN_METADATA = NEW_DATA_ROOT / 'train_metadata.csv'
VAL_METADATA = NEW_DATA_ROOT / 'val_metadata.csv'
EXTRACT_ROOT = NEW_DATA_ROOT / '_extracted'
OUTPUT_DIR = PROJECT_DIR / 'phora_paper_outputs_newdata_final'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 42
N_SPLITS = 3
N_JOBS = max(1, min(4, (os.cpu_count() or 4)//2))  # CT volumes are memory-heavy

# Feature switches
RUN_ARCHIVE_EXTRACTION = True
RUN_PYRADIOMICS = True
RUN_CONTEXT_OT = True
RUN_BURDEN_DISTILLATION = True

# Experiment switches

FAST_MODE = False

RUN_ARCH_ABLATIONS = True
RUN_FEATURE_ABLATIONS = True
RUN_REPEATABILITY = True
RUN_DOMAIN_ROBUSTNESS = True

# Keep these on
RUN_CONTEXT_OT = True
RUN_BURDEN_DISTILLATION = True
RUN_PYRADIOMICS = True   # now means your handcrafted IBSI-inspired radiomics


if FAST_MODE:
    SCREEN_K = 100
    ENSEMBLE_SEEDS = (17,)
    ROUTER_TRIALS = 900
    BOOTSTRAP_ITERS = 1500
else:
    SCREEN_K = 140
    ENSEMBLE_SEEDS = (17, 37, 61)
    ROUTER_TRIALS = 5000
    BOOTSTRAP_ITERS = 10000

print('PROJECT_DIR:', PROJECT_DIR)
print('NEW_DATA_ROOT:', NEW_DATA_ROOT)
print('FAST_MODE:', FAST_MODE, '| SCREEN_K:', SCREEN_K, '| ensemble:', ENSEMBLE_SEEDS)

PROJECT_DIR: e:\HCC_MICCAI26\PHORA_LI_NEW_DATA_PAPER_PACKAGE
NEW_DATA_ROOT: E:\HCC_MICCAI26\new_data
FAST_MODE: False | SCREEN_K: 140 | ensemble: (17, 37, 61)


## 2. Import the PHORA modules

Keep these beside the notebook:

- `phora_features.py`
- `phora_plus_v2.py`
- `extract_amplifai_pyradiomics.py` (only if `RUN_PYRADIOMICS=True`)

In [48]:
sys.path.insert(0, str(PROJECT_DIR))

from phora_features import (
    discover_cases, extract_case_features, extract_annotation_burdens
)
from phora_plus_v2 import (
    VALID_LABELS, ORDINAL_LABELS, SPECIAL_LABELS,
    fast_challenge_score, feature_groups_v2,
    cross_validate_phora_plus_v2,
    fit_predict_phora_plus_holdout_v2,
    run_holdout_baselines_v2,
    multiview_stack_holdout_v2,
    oracle_concept_holdout_v2,
    extract_context_ot_case,
    fit_phora_plus_full_v2,
    predict_phora_plus_full_v2,
    nested_screen, balanced_weights, xgb_classifier
)

print('Legal challenge labels:', VALID_LABELS)

Legal challenge labels: ['LR-1', 'LR-2', 'LR-3', 'LR-4', 'LR-5', 'LR-M', 'LR-TIV']


## 3. Unpack the four batch ZIPs once

The extractor is recursive, so it does not matter whether a ZIP expands as `batch_001/cases/...` or `batch_001/batch_001/cases/...`.

If you manually extracted the archives already, set `RUN_ARCHIVE_EXTRACTION=False` and set `EXTRACT_ROOT` to the common parent containing all extracted batches.

In [49]:
def find_batch_archives(root: Path):
    out=[]
    for p in root.iterdir():
        if p.is_file() and p.name.lower().startswith('batch_'):
            try:
                if zipfile.is_zipfile(p): out.append(p)
            except Exception:
                pass
    return sorted(out)

archives = find_batch_archives(NEW_DATA_ROOT)
print('ZIP archives:', [p.name for p in archives])

if RUN_ARCHIVE_EXTRACTION:
    if not archives:
        print('No batch ZIPs found; assuming EXTRACT_ROOT already contains extracted cases.')
    for zpath in archives:
        target = EXTRACT_ROOT / zpath.stem
        marker = target / '.phora_extracted.ok'
        if marker.exists():
            print('skip:', zpath.name)
            continue
        target.mkdir(parents=True, exist_ok=True)
        print('Extracting', zpath.name, '->', target)
        t0=time.time()
        with zipfile.ZipFile(zpath, 'r') as zf:
            zf.extractall(target)
        marker.write_text('ok', encoding='utf-8')
        print(f'  done in {(time.time()-t0)/60:.1f} min')

ZIP archives: ['batch_001.zip', 'batch_002.zip', 'batch_003.zip', 'batch_004.zip']
skip: batch_001.zip
skip: batch_002.zip
skip: batch_003.zip
skip: batch_004.zip


## 4. Load and audit the updated train/validation metadata

Important: the metadata field `aphe` is categorical in the updated CSV. We explicitly create two auxiliary targets:

\[
z_{A}=\mathbf 1[	ext{Non-rim APHE}], \qquad
z_R=\mathbf 1[	ext{Rim APHE}].
\]

They are **supervision targets only**, never input predictors.

In [50]:
assert TRAIN_METADATA.exists(), TRAIN_METADATA
assert VAL_METADATA.exists(), VAL_METADATA

train_raw = pd.read_csv(TRAIN_METADATA)
val_raw = pd.read_csv(VAL_METADATA)


def prepare_metadata(df, split):
    d=df.copy()
    d['case_id']=d['case_id'].astype(str)
    d['split']=split
    d['aphe_raw']=d['aphe']
    known=d['aphe_raw'].notna()
    d['rim_aphe']=np.where(known, (d['aphe_raw'].astype(str)=='Rim APHE').astype(float), np.nan)
    d['aphe']=np.where(known, (d['aphe_raw'].astype(str)=='Non-rim APHE').astype(float), np.nan)
    for c in ['washout_venous','washout_delayed','capsule_venous','capsule_delayed','max_diameter_mm','lesion']:
        if c in d: d[c]=pd.to_numeric(d[c],errors='coerce')
    return d

train_meta=prepare_metadata(train_raw,'train')
val_meta=prepare_metadata(val_raw,'val')

print('Raw split sizes:', len(train_meta), len(val_meta))
print('\nTrain labels:'); display(train_meta['lirads_score'].value_counts(dropna=False).to_frame('n'))
print('\nValidation labels:'); display(val_meta['lirads_score'].value_counts(dropna=False).to_frame('n'))

train_cls=train_meta[train_meta['lirads_score'].isin(VALID_LABELS)].reset_index(drop=True)
val_cls=val_meta[val_meta['lirads_score'].isin(VALID_LABELS)].reset_index(drop=True)
print('7-class lesion cases -> train:',len(train_cls),'validation:',len(val_cls))
print('Excluded train rows:', len(train_meta)-len(train_cls))

# Useful consistency audit.
inconsistent=train_meta[(train_meta['lirads_score'].astype(str)=='No lesion') & (train_meta['lesion']==1)]
if len(inconsistent):
    print('No-lesion/lesion-flag inconsistency (excluded anyway):')
    display(inconsistent[['case_id','batch_id','lesion','lirads_score']])

Raw split sizes: 531 59

Train labels:


,n
lirads_score,
LR-5,231
LR-M,115
No lesion,70
LR-TIV,57
LR-4,41
LR-3,13
LR-2,2
LR-1,2



Validation labels:


,n
lirads_score,
LR-5,27
LR-M,14
LR-TIV,7
LR-4,6
LR-3,3
LR-1,1
LR-2,1


7-class lesion cases -> train: 461 validation: 59
Excluded train rows: 70
No-lesion/lesion-flag inconsistency (excluded anyway):


,case_id,batch_id,lesion,lirads_score
178,CASE00197,batch_003,1,No lesion


## 5. Discover scored lesion cases after extraction

In [51]:
cases=discover_cases(EXTRACT_ROOT)
needed_ids=sorted(set(train_cls.case_id) | set(val_cls.case_id))
missing=sorted(set(needed_ids)-set(cases))
print('Discovered lesion case folders:',len(cases))
print('Needed 7-class cases:',len(needed_ids))
print('Missing needed cases:',len(missing))
if missing:
    print(missing[:20])
    raise FileNotFoundError('Some train/validation lesion cases were not discovered. Check ZIP extraction.')

Discovered lesion case folders: 521
Needed 7-class cases: 520
Missing needed cases: 0


# Part I — Inference-safe feature extraction

The test-time feature set must be computable using only multiphase CT + the supplied lesion mask.

## 6. Custom spatial/physiology features

Feature families include morphology, shell habitats, graph total variation / Dirichlet energy, semivariograms, hotspot displacement, phase-difference fields, trajectory curvature, spectral morphology, topology, and fractal descriptors.

In [52]:
from joblib import Parallel, delayed

SPATIAL_CSV=OUTPUT_DIR/'spatial_physiology_features.csv'
if SPATIAL_CSV.exists():
    spatial_df=pd.read_csv(SPATIAL_CSV)
else:
    spatial_df=pd.DataFrame(columns=['case_id'])

done=set(spatial_df.get('case_id',pd.Series(dtype=str)).astype(str))
todo=[cid for cid in needed_ids if cid not in done]
print('Spatial feature cases remaining:',len(todo))
if todo:
    rows=Parallel(n_jobs=N_JOBS,verbose=10)(
        delayed(extract_case_features)(cases[cid],cid,5) for cid in todo
    )
    spatial_df=pd.concat([spatial_df,pd.DataFrame(rows)],ignore_index=True)
    spatial_df=spatial_df.drop_duplicates('case_id',keep='last').sort_values('case_id')
    spatial_df.to_csv(SPATIAL_CSV,index=False)
print('spatial_df:',spatial_df.shape)

Spatial feature cases remaining: 0
spatial_df: (520, 425)


## 7. Optional IBSI-style PyRadiomics feature view

This adds shape, first-order, GLCM, GLRLM, GLSZM, GLDM, and NGTDM features independently for each available CT phase, plus phase-difference first-order descriptors.

It is useful both as a strong conventional radiomics baseline and as a complementary feature view.

In [53]:
import sys
import subprocess
import importlib
import importlib.util

print("Current notebook Python:")
print(sys.executable)

# Expected environment
EXPECTED = r"f:\CORI_Final\.venv\Scripts\python.exe"

if str(sys.executable).lower() != EXPECTED.lower():
    print("\nWARNING:")
    print("This notebook is NOT using the expected CORI_Final environment.")
    print("Expected:", EXPECTED)
    print("Actual:  ", sys.executable)
    print("\nIn VS Code: click the kernel name at the top-right and select:")
    print(EXPECTED)

# Install SimpleITK directly into whatever kernel is ACTUALLY running
if importlib.util.find_spec("SimpleITK") is None:
    print("\nSimpleITK missing in current kernel. Installing...")
    
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "SimpleITK==2.5.6",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    
    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            "SimpleITK installation failed. "
            "The real pip error is printed above."
        )

# Refresh Python's module cache
importlib.invalidate_caches()

# Verify
import SimpleITK as sitk

print("\n====================================")
print("SimpleITK READY")
print("====================================")
print("Python:", sys.executable)
print("SimpleITK:", sitk.Version_VersionString())
print("Location:", sitk.__file__)

Current notebook Python:
f:\CORI_Final\.venv\Scripts\python.exe

SimpleITK READY
Python: f:\CORI_Final\.venv\Scripts\python.exe
SimpleITK: 2.5.6
Location: f:\CORI_Final\.venv\Lib\site-packages\SimpleITK\__init__.py


In [54]:
# IBSI-inspired handcrafted radiomics (no PyRadiomics dependency)
# Produces `pyrad_df` so all downstream PHORA+ notebook cells remain unchanged.

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import math, traceback, warnings

import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage, stats

PYRAD_CSV = OUTPUT_DIR / "handcrafted_ibsi_radiomics.csv"
PYRAD_FAILURE_CSV = OUTPUT_DIR / "handcrafted_ibsi_radiomics_failures.csv"

RAD_SPACING_MM = 1.0
RAD_BIN_WIDTH_HU = 25.0
RAD_CT_MIN_HU = -1024.0
RAD_CT_MAX_HU = 3071.0
RAD_MIN_VOXELS = 10
RAD_CROP_PAD_MM = 5.0
RAD_WORKERS = max(1, min(int(N_JOBS), 2))   # Windows-safe / memory-safe

PHASES = ("ART", "VEN", "DEL", "DRY")
DIRS13 = [
    (1,0,0),(0,1,0),(0,0,1),
    (1,1,0),(1,-1,0),(1,0,1),(1,0,-1),(0,1,1),(0,1,-1),
    (1,1,1),(1,1,-1),(1,-1,1),(1,-1,-1),
]
NEIGH26 = [
    (z,y,x) for z in (-1,0,1) for y in (-1,0,1) for x in (-1,0,1)
    if (z,y,x) != (0,0,0)
]


def _safe_div(a, b):
    return float(a / b) if np.isfinite(b) and abs(float(b)) > 1e-12 else np.nan


def _finite(x):
    x = np.asarray(x, dtype=np.float64)
    return x[np.isfinite(x)]


def _discover_cases(root):
    out = {}
    for p in Path(root).rglob("lesion.nii.gz"):
        case_dir = p.parent.parent
        if (case_dir / "ct").is_dir():
            out[case_dir.name] = case_dir
    if not out:
        for p in Path(root).rglob("lesion.nii"):
            case_dir = p.parent.parent
            if (case_dir / "ct").is_dir():
                out[case_dir.name] = case_dir
    return out


def _find_phase(case_dir, case_id, phase):
    ct = case_dir / "ct"
    for suffix in (".nii.gz", ".nii"):
        p = ct / f"{case_id}_{phase}{suffix}"
        if p.exists():
            return p
    matches = sorted(ct.glob(f"*_{phase}.nii*"))
    return matches[0] if matches else None


def _read_binary_mask(path):
    m = sitk.ReadImage(str(path))
    return sitk.Cast(m > 0, sitk.sitkUInt8)


def _align_mask(mask, image):
    return sitk.Resample(
        mask, image, sitk.Transform(), sitk.sitkNearestNeighbor,
        0, sitk.sitkUInt8
    )


def _crop_to_mask(image, mask, pad_mm=5.0):
    a = sitk.GetArrayViewFromImage(mask) > 0
    if not np.any(a):
        raise ValueError("empty mask after alignment")
    zz, yy, xx = np.where(a)
    spacing = np.asarray(image.GetSpacing(), float)  # x,y,z
    pad_xyz = np.maximum(1, np.ceil(float(pad_mm) / spacing).astype(int))

    x0 = max(0, int(xx.min()) - int(pad_xyz[0]))
    y0 = max(0, int(yy.min()) - int(pad_xyz[1]))
    z0 = max(0, int(zz.min()) - int(pad_xyz[2]))
    x1 = min(image.GetSize()[0], int(xx.max()) + 1 + int(pad_xyz[0]))
    y1 = min(image.GetSize()[1], int(yy.max()) + 1 + int(pad_xyz[1]))
    z1 = min(image.GetSize()[2], int(zz.max()) + 1 + int(pad_xyz[2]))

    index = [x0, y0, z0]
    size = [x1-x0, y1-y0, z1-z0]
    return sitk.RegionOfInterest(image, size=size, index=index), sitk.RegionOfInterest(mask, size=size, index=index)


def _resample_isotropic(image, mask, spacing_mm=1.0):
    old_sp = np.asarray(image.GetSpacing(), float)
    old_sz = np.asarray(image.GetSize(), int)
    new_sp = np.array([spacing_mm]*3, float)
    new_sz = np.maximum(1, np.rint(old_sz * old_sp / new_sp).astype(int))

    def _resample(im, interp, default):
        f = sitk.ResampleImageFilter()
        f.SetOutputSpacing(tuple(new_sp.tolist()))
        f.SetSize([int(v) for v in new_sz])
        f.SetOutputOrigin(image.GetOrigin())
        f.SetOutputDirection(image.GetDirection())
        f.SetTransform(sitk.Transform())
        f.SetInterpolator(interp)
        f.SetDefaultPixelValue(default)
        return f.Execute(im)

    image_r = _resample(image, sitk.sitkBSpline, -1024.0)
    mask_r = _resample(mask, sitk.sitkNearestNeighbor, 0)
    return sitk.Cast(image_r, sitk.sitkFloat32), sitk.Cast(mask_r > 0, sitk.sitkUInt8)


def _prepare_phase(image_path, native_mask):
    img = sitk.ReadImage(str(image_path), sitk.sitkFloat32)
    m = _align_mask(native_mask, img)
    img, m = _crop_to_mask(img, m, RAD_CROP_PAD_MM)
    img, m = _resample_isotropic(img, m, RAD_SPACING_MM)
    arr = sitk.GetArrayFromImage(img).astype(np.float32)   # z,y,x
    roi = sitk.GetArrayFromImage(m) > 0
    roi &= np.isfinite(arr)
    if int(roi.sum()) < RAD_MIN_VOXELS:
        raise ValueError(f"ROI too small after resampling: {int(roi.sum())} voxels")
    return arr, roi


def _discretize_ct(arr, mask):
    """Fixed-bin-width discretization with a fixed CT origin for reproducibility."""
    q = np.zeros(arr.shape, dtype=np.int16)
    x = np.clip(arr[mask], RAD_CT_MIN_HU, RAD_CT_MAX_HU)
    bins = np.floor((x - RAD_CT_MIN_HU) / RAD_BIN_WIDTH_HU).astype(np.int16) + 1
    q[mask] = bins
    return q, int(bins.max())


def _shape_features(mask, spacing_zyx):
    n = int(mask.sum())
    if n == 0:
        return {}
    sp = np.asarray(spacing_zyx, float)
    voxel_volume = float(np.prod(sp))
    volume = n * voxel_volume

    # Surface area from exposed voxel faces; spacing-aware and deterministic.
    surface = 0.0
    for ax in range(3):
        pad = [(0,0)] * 3
        pad[ax] = (1,1)
        transitions = np.abs(np.diff(np.pad(mask.astype(np.int8), pad, mode="constant"), axis=ax)).sum()
        face_area = float(np.prod(np.delete(sp, ax)))
        surface += float(transitions) * face_area

    coords = np.argwhere(mask).astype(float) * sp[None, :]
    ext = coords.max(0) - coords.min(0) + sp
    eq_d = (6.0 * volume / np.pi) ** (1.0/3.0)
    sphericity = (np.pi ** (1.0/3.0) * (6.0*volume) ** (2.0/3.0) / surface) if surface > 0 else np.nan

    if n >= 4:
        eig = np.sort(np.linalg.eigvalsh(np.cov(coords, rowvar=False)))[::-1]
        elongation = np.sqrt(eig[1] / eig[0]) if eig[0] > 0 and eig[1] >= 0 else np.nan
        flatness = np.sqrt(eig[2] / eig[0]) if eig[0] > 0 and eig[2] >= 0 else np.nan
    else:
        elongation = flatness = np.nan

    return {
        "shape_VoxelCount": n,
        "shape_Volume_mm3": volume,
        "shape_SurfaceArea_mm2": surface,
        "shape_SurfaceVolumeRatio": _safe_div(surface, volume),
        "shape_Sphericity": float(sphericity),
        "shape_EquivalentDiameter_mm": float(eq_d),
        "shape_MaxBBoxDiameter_mm": float(np.max(ext)),
        "shape_BBoxDiagonal_mm": float(np.linalg.norm(ext)),
        "shape_Elongation": float(elongation),
        "shape_Flatness": float(flatness),
    }


def _firstorder(arr, mask):
    x = _finite(arr[mask])
    if x.size == 0:
        return {}
    p10, p25, p75, p90 = np.percentile(x, [10,25,75,90])
    med = float(np.median(x))
    mad = float(np.median(np.abs(x-med)))
    trimmed = x[(x >= p10) & (x <= p90)]
    rmed = float(np.median(trimmed)) if trimmed.size else np.nan
    rmad = float(np.median(np.abs(trimmed-rmed))) if trimmed.size else np.nan
    hist, _ = np.histogram(x, bins=min(128, max(16, int(np.sqrt(x.size)))))
    p = hist[hist > 0].astype(float)
    p /= p.sum()
    mu = float(np.mean(x)); sd = float(np.std(x))
    return {
        "original_firstorder_Mean": mu,
        "original_firstorder_Median": med,
        "original_firstorder_Minimum": float(np.min(x)),
        "original_firstorder_Maximum": float(np.max(x)),
        "original_firstorder_Range": float(np.ptp(x)),
        "original_firstorder_Variance": float(np.var(x)),
        "original_firstorder_StandardDeviation": sd,
        "original_firstorder_Skewness": float(stats.skew(x, bias=False)) if x.size > 2 else np.nan,
        "original_firstorder_Kurtosis": float(stats.kurtosis(x, bias=False, fisher=False)) if x.size > 3 else np.nan,
        "original_firstorder_10Percentile": float(p10),
        "original_firstorder_25Percentile": float(p25),
        "original_firstorder_75Percentile": float(p75),
        "original_firstorder_90Percentile": float(p90),
        "original_firstorder_InterquartileRange": float(p75-p25),
        "original_firstorder_MeanAbsoluteDeviation": float(np.mean(np.abs(x-mu))),
        "original_firstorder_MedianAbsoluteDeviation": mad,
        "original_firstorder_RobustMedianAbsoluteDeviation": rmad,
        "original_firstorder_Energy": float(np.sum(x*x)),
        "original_firstorder_RootMeanSquared": float(np.sqrt(np.mean(x*x))),
        "original_firstorder_Entropy": float(-np.sum(p*np.log2(p))),
        "original_firstorder_CoefficientOfVariation": _safe_div(sd, abs(mu)),
    }


def _neighbor(arr, offset, fill=0):
    """out[v] = arr[v + offset], with constant fill outside."""
    out = np.full_like(arr, fill)
    src, dst = [], []
    for n, o in zip(arr.shape, offset):
        if o >= 0:
            src.append(slice(o, n)); dst.append(slice(0, n-o))
        else:
            src.append(slice(0, n+o)); dst.append(slice(-o, n))
    out[tuple(dst)] = arr[tuple(src)]
    return out


def _glcm(q, mask, G):
    P = np.zeros((G,G), dtype=np.float64)
    Z,Y,X = q.shape
    for dz,dy,dx in DIRS13:
        z0,z1=max(0,-dz),min(Z,Z-dz)
        y0,y1=max(0,-dy),min(Y,Y-dy)
        x0,x1=max(0,-dx),min(X,X-dx)
        a=q[z0:z1,y0:y1,x0:x1]
        b=q[z0+dz:z1+dz,y0+dy:y1+dy,x0+dx:x1+dx]
        m=mask[z0:z1,y0:y1,x0:x1] & mask[z0+dz:z1+dz,y0+dy:y1+dy,x0+dx:x1+dx]
        if not np.any(m):
            continue
        ai=a[m]-1; bi=b[m]-1
        np.add.at(P,(ai,bi),1)
        np.add.at(P,(bi,ai),1)  # symmetric aggregation
    if P.sum() == 0:
        return {}
    P /= P.sum()
    i=np.arange(1,G+1,dtype=float)[:,None]
    j=np.arange(1,G+1,dtype=float)[None,:]
    d=i-j
    mui=float((P*i).sum()); muj=float((P*j).sum())
    si=float(np.sqrt((P*(i-mui)**2).sum())); sj=float(np.sqrt((P*(j-muj)**2).sum()))
    nz=P[P>0]
    asm=float(np.sum(P*P))
    return {
        "original_glcm_Contrast": float(np.sum(P*d*d)),
        "original_glcm_Dissimilarity": float(np.sum(P*np.abs(d))),
        "original_glcm_Idm": float(np.sum(P/(1.0+d*d))),
        "original_glcm_JointEnergy": asm,
        "original_glcm_Energy": float(np.sqrt(asm)),
        "original_glcm_Correlation": _safe_div(np.sum(P*(i-mui)*(j-muj)), si*sj),
        "original_glcm_JointEntropy": float(-np.sum(nz*np.log2(nz))),
        "original_glcm_JointAverage": mui,
        "original_glcm_ClusterShade": float(np.sum(P*(i+j-mui-muj)**3)),
        "original_glcm_ClusterProminence": float(np.sum(P*(i+j-mui-muj)**4)),
    }


def _matrix_features(P, nvox, kind):
    total=float(P.sum())
    if total <= 0:
        return {}
    pg=P.sum(axis=1); ps=P.sum(axis=0)
    g=np.arange(1,P.shape[0]+1,dtype=float)[:,None]
    s=np.arange(1,P.shape[1]+1,dtype=float)[None,:]

    if kind == "rl":
        return {
            "original_glrlm_ShortRunEmphasis": float(np.sum(P/s**2)/total),
            "original_glrlm_LongRunEmphasis": float(np.sum(P*s**2)/total),
            "original_glrlm_GrayLevelNonUniformity": float(np.sum(pg**2)/total),
            "original_glrlm_GrayLevelNonUniformityNormalized": float(np.sum(pg**2)/total**2),
            "original_glrlm_RunLengthNonUniformity": float(np.sum(ps**2)/total),
            "original_glrlm_RunLengthNonUniformityNormalized": float(np.sum(ps**2)/total**2),
            "original_glrlm_RunPercentage": _safe_div(total,nvox),
            "original_glrlm_LowGrayLevelRunEmphasis": float(np.sum(P/g**2)/total),
            "original_glrlm_HighGrayLevelRunEmphasis": float(np.sum(P*g**2)/total),
            "original_glrlm_ShortRunLowGrayLevelEmphasis": float(np.sum(P/(s**2*g**2))/total),
            "original_glrlm_ShortRunHighGrayLevelEmphasis": float(np.sum(P*g**2/s**2)/total),
            "original_glrlm_LongRunLowGrayLevelEmphasis": float(np.sum(P*s**2/g**2)/total),
            "original_glrlm_LongRunHighGrayLevelEmphasis": float(np.sum(P*s**2*g**2)/total),
        }
    if kind == "sz":
        return {
            "original_glszm_SmallAreaEmphasis": float(np.sum(P/s**2)/total),
            "original_glszm_LargeAreaEmphasis": float(np.sum(P*s**2)/total),
            "original_glszm_GrayLevelNonUniformity": float(np.sum(pg**2)/total),
            "original_glszm_GrayLevelNonUniformityNormalized": float(np.sum(pg**2)/total**2),
            "original_glszm_SizeZoneNonUniformity": float(np.sum(ps**2)/total),
            "original_glszm_SizeZoneNonUniformityNormalized": float(np.sum(ps**2)/total**2),
            "original_glszm_ZonePercentage": _safe_div(total,nvox),
            "original_glszm_LowGrayLevelZoneEmphasis": float(np.sum(P/g**2)/total),
            "original_glszm_HighGrayLevelZoneEmphasis": float(np.sum(P*g**2)/total),
            "original_glszm_SmallAreaLowGrayLevelEmphasis": float(np.sum(P/(s**2*g**2))/total),
            "original_glszm_SmallAreaHighGrayLevelEmphasis": float(np.sum(P*g**2/s**2)/total),
            "original_glszm_LargeAreaLowGrayLevelEmphasis": float(np.sum(P*s**2/g**2)/total),
            "original_glszm_LargeAreaHighGrayLevelEmphasis": float(np.sum(P*s**2*g**2)/total),
        }
    if kind == "dm":
        return {
            "original_gldm_SmallDependenceEmphasis": float(np.sum(P/s**2)/total),
            "original_gldm_LargeDependenceEmphasis": float(np.sum(P*s**2)/total),
            "original_gldm_GrayLevelNonUniformity": float(np.sum(pg**2)/total),
            "original_gldm_GrayLevelNonUniformityNormalized": float(np.sum(pg**2)/total**2),
            "original_gldm_DependenceNonUniformity": float(np.sum(ps**2)/total),
            "original_gldm_DependenceNonUniformityNormalized": float(np.sum(ps**2)/total**2),
            "original_gldm_LowGrayLevelEmphasis": float(np.sum(P/g**2)/total),
            "original_gldm_HighGrayLevelEmphasis": float(np.sum(P*g**2)/total),
            "original_gldm_SmallDependenceLowGrayLevelEmphasis": float(np.sum(P/(s**2*g**2))/total),
            "original_gldm_SmallDependenceHighGrayLevelEmphasis": float(np.sum(P*g**2/s**2)/total),
            "original_gldm_LargeDependenceLowGrayLevelEmphasis": float(np.sum(P*s**2/g**2)/total),
            "original_gldm_LargeDependenceHighGrayLevelEmphasis": float(np.sum(P*s**2*g**2)/total),
        }
    return {}


def _glrlm(q, mask, G):
    counts={}; Z,Y,X=q.shape; nvox=int(mask.sum())
    for dz,dy,dx in DIRS13:
        prev=_neighbor(mask,(-dz,-dy,-dx),False)
        starts=np.argwhere(mask & ~prev)
        for z,y,x in starts:
            z=int(z); y=int(y); x=int(x)
            cz,cy,cx=z,y,x; cur=int(q[z,y,x]); run=0
            while 0<=cz<Z and 0<=cy<Y and 0<=cx<X and mask[cz,cy,cx]:
                g=int(q[cz,cy,cx])
                if g == cur:
                    run += 1
                else:
                    counts[(cur,run)] = counts.get((cur,run),0) + 1
                    cur=g; run=1
                cz+=dz; cy+=dy; cx+=dx
            counts[(cur,run)] = counts.get((cur,run),0) + 1
    if not counts:
        return {}
    S=max(r for _,r in counts)
    P=np.zeros((G,S),float)
    for (g,r),v in counts.items():
        P[g-1,r-1]+=v
    return _matrix_features(P,nvox,"rl")


def _glszm(q, mask, G):
    counts={}; structure=np.ones((3,3,3),np.uint8)
    for g in np.unique(q[mask]):
        lab,n=ndimage.label(mask & (q==g), structure=structure)
        if n == 0:
            continue
        sizes=np.bincount(lab.ravel())[1:]
        for s in sizes:
            counts[(int(g),int(s))]=counts.get((int(g),int(s)),0)+1
    if not counts:
        return {}
    S=max(s for _,s in counts)
    P=np.zeros((G,S),float)
    for (g,s),v in counts.items():
        P[g-1,s-1]+=v
    return _matrix_features(P,int(mask.sum()),"sz")


def _gldm(q, mask, G, alpha=0):
    dep=np.ones(q.shape,np.int16)
    Z,Y,X=q.shape
    for dz,dy,dx in NEIGH26:
        z0,z1=max(0,-dz),min(Z,Z-dz)
        y0,y1=max(0,-dy),min(Y,Y-dy)
        x0,x1=max(0,-dx),min(X,X-dx)
        a=q[z0:z1,y0:y1,x0:x1]
        b=q[z0+dz:z1+dz,y0+dy:y1+dy,x0+dx:x1+dx]
        m=mask[z0:z1,y0:y1,x0:x1] & mask[z0+dz:z1+dz,y0+dy:y1+dy,x0+dx:x1+dx]
        dep[z0:z1,y0:y1,x0:x1] += (m & (np.abs(a-b)<=alpha)).astype(np.int16)
    gs=q[mask].astype(int); ds=dep[mask].astype(int)
    P=np.zeros((G,int(ds.max())),float)
    np.add.at(P,(gs-1,ds-1),1)
    return _matrix_features(P,int(mask.sum()),"dm")


def _ngtdm(q, mask, G):
    kernel=np.ones((3,3,3),float); kernel[1,1,1]=0
    cnt=ndimage.convolve(mask.astype(float),kernel,mode="constant",cval=0.0)
    sm=ndimage.convolve((q*mask).astype(float),kernel,mode="constant",cval=0.0)
    valid=mask & (cnt>0)
    if not np.any(valid):
        return {}
    avg=np.zeros(q.shape,float); avg[valid]=sm[valid]/cnt[valid]
    n=np.zeros(G,float); s=np.zeros(G,float)
    for g in range(1,G+1):
        m=valid & (q==g)
        n[g-1]=m.sum()
        if n[g-1] > 0:
            s[g-1]=np.abs(g-avg[m]).sum()
    Nv=float(n.sum())
    if Nv <= 0:
        return {}
    p=n/Nv; idx=np.where(n>0)[0]; gi=np.arange(1,G+1,dtype=float)
    coarseness=1.0/(np.sum(p*s)+1e-12)
    denom=max(len(idx)*(len(idx)-1),1)
    contrast=float(np.sum((p[:,None]*p[None,:])*(gi[:,None]-gi[None,:])**2)/denom * (s.sum()/Nv))
    busy_den=sum(abs(gi[i]*p[i]-gi[j]*p[j]) for i in idx for j in idx)
    busyness=float(np.sum(p*s)/(busy_den+1e-12))
    complexity=0.0; strength_num=0.0
    for i in idx:
        for j in idx:
            if i == j: continue
            complexity += abs(gi[i]-gi[j])*(p[i]*s[i]+p[j]*s[j])/(p[i]+p[j]+1e-12)
            strength_num += (p[i]+p[j])*(gi[i]-gi[j])**2
    return {
        "original_ngtdm_Coarseness": float(coarseness),
        "original_ngtdm_Contrast": contrast,
        "original_ngtdm_Busyness": busyness,
        "original_ngtdm_Complexity": float(complexity/Nv),
        "original_ngtdm_Strength": float(strength_num/(s.sum()+1e-12)),
    }


def _texture_features(arr, mask):
    q,G=_discretize_ct(arr,mask)
    out={}
    out.update(_glcm(q,mask,G))
    out.update(_glrlm(q,mask,G))
    out.update(_glszm(q,mask,G))
    out.update(_gldm(q,mask,G,alpha=0))
    out.update(_ngtdm(q,mask,G))
    return out


def _native_shape(native_mask):
    a=sitk.GetArrayFromImage(native_mask)>0
    spacing_zyx=tuple(reversed(native_mask.GetSpacing()))
    return _shape_features(a,spacing_zyx)


def _extract_case(case_id, case_dir):
    row={"case_id":str(case_id)}; failures=[]
    lesion_path=case_dir/"annotations"/"lesion.nii.gz"
    if not lesion_path.exists():
        lesion_path=case_dir/"annotations"/"lesion.nii"
    try:
        native_mask=_read_binary_mask(lesion_path)
        row.update(_native_shape(native_mask))
    except Exception as e:
        return row,[{"case_id":case_id,"component":"shape","error":repr(e)}]

    for phase in PHASES:
        p=_find_phase(case_dir,case_id,phase)
        row[f"phase_present_{phase}"]=int(p is not None)
        if p is None:
            continue
        try:
            arr,mask=_prepare_phase(p,native_mask)
            row[f"{phase}_roi_voxels"]=int(mask.sum())
            feats={}
            feats.update(_firstorder(arr,mask))
            feats.update(_texture_features(arr,mask))
            for k,v in feats.items():
                row[f"{phase}_{k}"]=float(v) if np.isscalar(v) else np.nan
        except Exception as e:
            failures.append({"case_id":case_id,"component":phase,"error":repr(e)})
    return row,failures


def _add_phase_deltas(df):
    pairs=(("ART","DRY"),("VEN","ART"),("DEL","ART"),("DEL","VEN"))
    suffixes=set()
    for c in df.columns:
        for ph in PHASES:
            prefix=f"{ph}_original_firstorder_"
            if c.startswith(prefix):
                suffixes.add(c[len(ph)+1:])
    add={}
    for left,right in pairs:
        for suf in suffixes:
            a=f"{left}_{suf}"; b=f"{right}_{suf}"
            if a in df.columns and b in df.columns:
                add[f"DELTA_{left}_MINUS_{right}_{suf}"] = pd.to_numeric(df[a],errors="coerce") - pd.to_numeric(df[b],errors="coerce")
    if add:
        df=pd.concat([df,pd.DataFrame(add,index=df.index)],axis=1)
    return df


# -------------------------------
# Execute + cache
# -------------------------------
if RUN_PYRADIOMICS:
    if PYRAD_CSV.exists():
        print("Loading cached handcrafted radiomics:", PYRAD_CSV)
        pyrad_df=pd.read_csv(PYRAD_CSV)
    else:
        cases=_discover_cases(EXTRACT_ROOT)
        targets=[str(x) for x in needed_ids if str(x) in cases]
        missing=sorted(set(map(str,needed_ids))-set(targets))
        print(f"Discovered {len(cases)} cases; extracting {len(targets)} requested cases")
        if missing:
            print("WARNING: requested IDs not found:", missing[:10], "..." if len(missing)>10 else "")

        rows=[]; failures=[]
        with ThreadPoolExecutor(max_workers=RAD_WORKERS) as ex:
            fut={ex.submit(_extract_case,cid,cases[cid]):cid for cid in targets}
            for i,f in enumerate(as_completed(fut),1):
                cid=fut[f]
                try:
                    row,errs=f.result(); rows.append(row); failures.extend(errs)
                except Exception as e:
                    rows.append({"case_id":cid})
                    failures.append({"case_id":cid,"component":"worker","error":traceback.format_exc()[-4000:]})
                if i==1 or i%25==0 or i==len(fut):
                    print(f"Radiomics {i}/{len(fut)}")

        pyrad_df=pd.DataFrame(rows).sort_values("case_id").reset_index(drop=True)
        pyrad_df=_add_phase_deltas(pyrad_df)
        pyrad_df.to_csv(PYRAD_CSV,index=False)
        pd.DataFrame(failures,columns=["case_id","component","error"]).to_csv(PYRAD_FAILURE_CSV,index=False)
        print("Saved:",PYRAD_CSV)
        print("Failures:",len(failures),"->",PYRAD_FAILURE_CSV)

    pyrad_df["case_id"]=pyrad_df["case_id"].astype(str)
    pyrad_df=pyrad_df[pyrad_df.case_id.isin(set(map(str,needed_ids)))].reset_index(drop=True)
else:
    pyrad_df=pd.DataFrame({"case_id":list(map(str,needed_ids))})

print("pyrad_df:",pyrad_df.shape)
print("Handcrafted radiomics feature columns:",max(0,pyrad_df.shape[1]-1))
display(pyrad_df.head())

Loading cached handcrafted radiomics: e:\HCC_MICCAI26\PHORA_LI_NEW_DATA_PAPER_PACKAGE\phora_paper_outputs_newdata_final\handcrafted_ibsi_radiomics.csv
pyrad_df: (520, 399)
Handcrafted radiomics feature columns: 398


,case_id,shape_VoxelCount,shape_Volume_mm3,shape_SurfaceArea_mm2,shape_SurfaceVolumeRatio,shape_Sphericity,shape_EquivalentDiameter_mm,shape_MaxBBoxDiameter_mm,shape_BBoxDiagonal_mm,shape_Elongation,...,DELTA_DEL_MINUS_VEN_original_firstorder_Range,DELTA_DEL_MINUS_VEN_original_firstorder_Entropy,DELTA_DEL_MINUS_VEN_original_firstorder_Mean,DELTA_DEL_MINUS_VEN_original_firstorder_MedianAbsoluteDeviation,DELTA_DEL_MINUS_VEN_original_firstorder_Minimum,DELTA_DEL_MINUS_VEN_original_firstorder_RobustMedianAbsoluteDeviation,DELTA_DEL_MINUS_VEN_original_firstorder_Median,DELTA_DEL_MINUS_VEN_original_firstorder_InterquartileRange,DELTA_DEL_MINUS_VEN_original_firstorder_CoefficientOfVariation,DELTA_DEL_MINUS_VEN_original_firstorder_75Percentile
0,CASE00001,94878,113389.236909,19592.493958,0.172790,0.578241,60.051575,68.449195,115.655220,0.778968,...,-90.436535,0.284629,-23.527481,-1.995792,43.125515,-1.522964,-23.742245,-3.998736,0.027560,-25.723665
1,CASE00002,9326,4553.711005,3216.386738,0.706322,0.413072,20.564695,31.250000,41.681148,0.719355,...,-95.333626,0.047140,-23.893620,-7.075842,-4.603806,-5.868484,-15.890043,-7.265651,29.311563,-24.684776
2,CASE00003,49700,74702.973611,13187.724704,0.176535,0.650438,52.253224,61.255888,93.353772,0.781568,...,342.902954,-1.085439,-19.357589,-0.001257,-353.159378,0.032269,-18.147511,0.048971,0.112440,-18.272297
3,CASE00004,96804,47267.578829,19722.470815,0.417252,0.320552,44.859550,85.937500,105.993048,0.459707,...,-34.900116,-0.113801,-1.011643,-4.631317,4.092758,-3.768259,0.584684,-9.069547,-0.090458,-4.020096
4,CASE00005,14616,7136.718856,3328.554707,0.466398,0.538552,23.887278,36.718750,50.379635,0.897989,...,62.530655,-0.638264,-31.201274,-2.622039,-112.822113,-1.854234,-28.773870,-4.530562,0.411635,-32.680558


## 8. Local liver-reference + optimal-transport features

Instead of using absolute lesion HU alone, define a local background-corrected contrast

\[
C_p=\operatorname{median}_{v\in\Omega}I_p(v)-
\operatorname{median}_{v\in R}I_p(v),
\]

where \(R\) is a 3–12 mm perilesional reference ring.

The extractor also computes 1-D Wasserstein distances, Jensen–Shannon divergence, Moran's \(I\), Geary's \(C\), hotspot topology, and a perilesional high-enhancement contact surrogate intended to help LR-TIV.

In [55]:
CONTEXT_CSV=OUTPUT_DIR/'context_ot_features.csv'
if RUN_CONTEXT_OT:
    if CONTEXT_CSV.exists():
        context_df=pd.read_csv(CONTEXT_CSV)
    else:
        context_df=pd.DataFrame(columns=['case_id'])
    done=set(context_df.get('case_id',pd.Series(dtype=str)).astype(str))
    todo=[cid for cid in needed_ids if cid not in done]
    print('Context/OT cases remaining:',len(todo))
    if todo:
        rows=Parallel(n_jobs=N_JOBS,verbose=10)(
            delayed(extract_context_ot_case)(cases[cid],cid) for cid in todo
        )
        context_df=pd.concat([context_df,pd.DataFrame(rows)],ignore_index=True)
        context_df=context_df.drop_duplicates('case_id',keep='last').sort_values('case_id')
        context_df.to_csv(CONTEXT_CSV,index=False)
else:
    context_df=pd.DataFrame({'case_id':needed_ids})
print('context_df:',context_df.shape)

Context/OT cases remaining: 520


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:   17.6s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   23.6s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:   35.9s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:   58.0s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:  1.4min
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  1.6min
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:  2.0min
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:  2.2min
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:  2.9min
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:  3.3min
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:  3.9min
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:  4.5min
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:  5.3min
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:  5.9min
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:  6.7min
[Parallel(

context_df: (520, 79)


## 9. Training-only spatial annotation burden distillation

For an annotation mask \(A_k\) and lesion mask \(\Omega\), define

\[
b_k=
rac{|A_k\cap\Omega|}{|\Omega|}.
\]

The true \(b_k\) values are **never used at inference**. PHORA+ learns cross-fitted regressors \(\hat b_k=f_k(x)\) from inference-safe CT features, then supplies predicted burden latents to the hierarchy. This turns the voxel annotations into richer soft supervision than a binary biomarker label alone.

In [56]:
BURDEN_CSV=OUTPUT_DIR/'annotation_burdens.csv'
if RUN_BURDEN_DISTILLATION:
    if BURDEN_CSV.exists():
        burden_df=pd.read_csv(BURDEN_CSV)
    else:
        burden_df=pd.DataFrame(columns=['case_id'])
    done=set(burden_df.get('case_id',pd.Series(dtype=str)).astype(str))
    todo=[cid for cid in needed_ids if cid not in done]
    print('Burden cases remaining:',len(todo))
    if todo:
        rows=Parallel(n_jobs=N_JOBS,verbose=10)(
            delayed(extract_annotation_burdens)(cases[cid],cid) for cid in todo
        )
        burden_df=pd.concat([burden_df,pd.DataFrame(rows)],ignore_index=True)
        burden_df=burden_df.drop_duplicates('case_id',keep='last').sort_values('case_id')
        burden_df.to_csv(BURDEN_CSV,index=False)
else:
    burden_df=pd.DataFrame({'case_id':needed_ids})
print('burden_df:',burden_df.shape)
display(burden_df.notna().mean().sort_values(ascending=False).to_frame('available_fraction').head(10))

Burden cases remaining: 520


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    2.1s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    3.0s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    4.0s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    5.4s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    6.2s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    7.2s
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:    8.8s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:   10.1s
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:   12.0s
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:   13.5s
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:   16.1s
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:   18.1s
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:   20.8s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   22.3s
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:   25.3s
[Parallel(

burden_df: (520, 6)


,available_fraction
case_id,1.000000
burden_aphe,0.711538
burden_washout_venous,0.409615
burden_washout_delayed,0.382692
burden_capsule_venous,0.190385
burden_capsule_delayed,0.125000


## 10. Merge modeling tables and perform leakage audit

In [57]:
def merge_features(meta):
    d=meta.copy()
    for table in [spatial_df,pyrad_df,context_df,burden_df]:
        overlap=[c for c in table.columns if c!='case_id' and c in d.columns]
        if overlap: table=table.drop(columns=overlap)
        d=d.merge(table,on='case_id',how='left',validate='one_to_one')
    return d

train_model=merge_features(train_cls)
val_model=merge_features(val_cls)

META_BANNED=set([
    'batch_id','case_id','split','lirads_score','lesion','max_diameter_mm','aphe_raw',
    'aphe','rim_aphe','washout_venous','washout_delayed','capsule_venous','capsule_delayed'
])
feature_cols=[]
for c in train_model.columns:
    if c in META_BANNED or c.startswith('burden_'): continue
    if pd.api.types.is_numeric_dtype(train_model[c]): feature_cols.append(c)

print('Train modeling rows:',train_model.shape,'Validation:',val_model.shape)
print('Inference-safe candidate features:',len(feature_cols))
print('Burden targets:',[c for c in train_model.columns if c.startswith('burden_')])
assert not any(c in feature_cols for c in META_BANNED)
assert not any(c.startswith('burden_') for c in feature_cols)

groups=feature_groups_v2(feature_cols)
display(pd.DataFrame({'feature_group':groups.keys(),'n_features':[len(v) for v in groups.values()]}))

Train modeling rows: (461, 918) Validation: (59, 918)
Inference-safe candidate features: 900
Burden targets: ['burden_aphe', 'burden_washout_venous', 'burden_washout_delayed', 'burden_capsule_venous', 'burden_capsule_delayed']


,feature_group,n_features
0,Morphology,37
1,Custom kinetics,356
2,Spatial fields,76
3,Context + OT,78
4,PyRadiomics,380


# Part II — Evaluation protocol

### Primary scientific protocol

- **Development / ablation:** only the 7-class cases from `train_metadata.csv`.
- **Final holdout:** train on the full training lesion cohort and evaluate once on `val_metadata.csv`.
- **Final challenge retraining:** only after all design decisions are frozen, combine train + validation and refit.

This avoids using the official validation split to repeatedly tune architecture choices.

In [58]:
def add_common_metrics(gt,pred):
    m=fast_challenge_score(gt,pred)
    m['accuracy']=accuracy_score(gt,pred)
    m['macro_f1']=f1_score(gt,pred,labels=VALID_LABELS,average='macro',zero_division=0)
    return m

print('Train class counts:')
display(train_model['lirads_score'].value_counts().reindex(VALID_LABELS).fillna(0).astype(int).to_frame('n'))
print('Validation class counts:')
display(val_model['lirads_score'].value_counts().reindex(VALID_LABELS).fillna(0).astype(int).to_frame('n'))

Train class counts:


,n
lirads_score,
LR-1,2
LR-2,2
LR-3,13
LR-4,41
LR-5,231
LR-M,115
LR-TIV,57


Validation class counts:


,n
lirads_score,
LR-1,1
LR-2,1
LR-3,3
LR-4,6
LR-5,27
LR-M,14
LR-TIV,7


## 11. Strong conventional baselines — official validation holdout

These all use training-only feature screening and never see validation labels during fitting.

In [59]:
t0=time.time()
baseline_table, baseline_preds, baseline_selected = run_holdout_baselines_v2(
    train_model,val_model,feature_cols,screen_k=SCREEN_K,seed=SEED
)
display(baseline_table)
baseline_table.to_csv(OUTPUT_DIR/'table_baselines_validation.csv',index=False)
print('baseline minutes:',(time.time()-t0)/60)

,Method,final_score,adjusted_qwk,special_category_recognition,accuracy,macro_f1
4,Extra Trees,0.762708,7.851852e-01,0.635338,0.644068,0.400319
3,Random Forest,0.705119,7.090301e-01,0.682957,0.644068,0.279381
1,Elastic-net logistic,0.689854,7.021277e-01,0.620301,0.576271,0.457360
6,Flat XGBoost,0.686617,6.857143e-01,0.691729,0.627119,0.282831
2,RBF-SVM,0.280690,2.322581e-01,0.555138,0.389831,0.228319
5,HistGradientBoosting,0.200213,1.134752e-01,0.691729,0.644068,0.279967
0,Majority class,0.050000,3.720930e-09,0.333333,0.457627,0.089701


baseline minutes: 0.33015509049097697


## 12. Simple hierarchy baseline

This isolates the value of **hierarchical decomposition alone**: no latent concepts, no factorized gate, no all-threshold expert, no clinical constraints.

In [60]:
simple_hier=fit_predict_phora_plus_holdout_v2(
    train_model,val_model,feature_cols,seed=SEED,screen_k=SCREEN_K,
    ensemble_seeds=ENSEMBLE_SEEDS,router_trials=ROUTER_TRIALS,
    use_concepts=False,use_factor_gate=False,use_all_threshold=False,
    use_screen=True,use_clinical_constraints=False
)
print(simple_hier['metrics'])

{'final_score': np.float64(0.6922077932051443), 'adjusted_qwk': np.float64(0.703349283470006), 'special_category_recognition': 0.6290726817042606}


## 13. PHORA+ v2 — frozen primary method on the official validation holdout

Key model:

\[
P(G\mid x),\qquad G\in\{	ext{ordinal},	ext{LR-M},	ext{LR-TIV}\},
\]

with a factorized special gate and an ordinal all-threshold expert

\[
q_t(x)=P(Y>t\mid x),\quad t=1,\ldots,4,
\]

projected to satisfy \(q_1\ge q_2\ge q_3\ge q_4\).

The final router is tuned only using cross-fitted predictions within the training split.

In [61]:
t0=time.time()
phora2=fit_predict_phora_plus_holdout_v2(
    train_model,val_model,feature_cols,seed=SEED,screen_k=SCREEN_K,
    ensemble_seeds=ENSEMBLE_SEEDS,router_trials=ROUTER_TRIALS,
    use_concepts=True,use_factor_gate=True,use_all_threshold=True,
    use_screen=True,use_clinical_constraints=True
)
print('PHORA+ v2:',phora2['metrics'])
print('minutes:',(time.time()-t0)/60)
phora2['prediction'].to_csv(OUTPUT_DIR/'validation_predictions_phora_plus_v2.csv',index=False)
print('Selected inference features:',len(phora2['selected_features']))
print('Router:',phora2['router']['params'])

PHORA+ v2: {'final_score': np.float64(0.7178622326569002), 'adjusted_qwk': np.float64(0.737732657614929), 'special_category_recognition': 0.6052631578947368}
minutes: 0.4784015734990438
Selected inference features: 140
Router: (0.25, 0.5, 1.3728349262146222, 3.008622115550845, 3.6930915394128876, 4.182945284091689, 0.3266953429886349, 0.6993298967900473, 0.08039821961617846, 0.2912981981082327, 0.9709756015689236)


## 14. Cross-fitted multi-view stacking baseline

Each biological view is trained independently; out-of-fold 7-class probabilities are concatenated and fused by a class-balanced meta-learner. This tests whether late evidence fusion beats early concatenation of all radiomics.

In [62]:
# Patch multiview_stack_holdout_v2 for newer scikit-learn versions

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

def multiview_stack_holdout_v2(
    train_df,
    test_df,
    feature_cols,
    n_splits=3,
    seed=42,
    per_view_k=60,
):
    """
    Cross-fitted multi-view stacking compatible with newer scikit-learn.
    """

    # ---------------------------------------------------------
    # Feature views
    # ---------------------------------------------------------
    groups = feature_groups_v2(feature_cols)

    y = train_df["lirads_score"].astype(str).to_numpy()

    label_to_int = {c: i for i, c in enumerate(VALID_LABELS)}
    yi = np.array([label_to_int[z] for z in y], dtype=int)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    oof_blocks = []
    test_blocks = []
    used_groups = {}

    # ---------------------------------------------------------
    # Train one XGBoost expert per feature view
    # ---------------------------------------------------------
    for view_name, cols in groups.items():

        cols = [c for c in cols if c in train_df.columns]

        if len(cols) == 0:
            continue

        # Select features using TRAINING DATA ONLY
        selected = nested_screen(
            train_df,
            cols,
            k=min(per_view_k, len(cols)),
            seed=seed,
        )

        if len(selected) == 0:
            continue

        used_groups[view_name] = selected

        X = (
            train_df[selected]
            .apply(pd.to_numeric, errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .to_numpy(np.float32)
        )

        Xt = (
            test_df[selected]
            .apply(pd.to_numeric, errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .to_numpy(np.float32)
        )

        oof_prob = np.zeros(
            (len(train_df), len(VALID_LABELS)),
            dtype=np.float32,
        )

        test_prob = np.zeros(
            (len(test_df), len(VALID_LABELS)),
            dtype=np.float32,
        )

        for fold, (tr, va) in enumerate(skf.split(X, yi)):

            model = xgb_classifier(
                n_classes=len(VALID_LABELS),
                seed=seed + fold + 100,
            )

            sw = balanced_weights(
                yi[tr],
                power=0.60,
            )

            model.fit(
                X[tr],
                yi[tr],
                sample_weight=sw,
            )

            oof_prob[va] = model.predict_proba(X[va])

            test_prob += (
                model.predict_proba(Xt) / n_splits
            )

        oof_blocks.append(oof_prob)
        test_blocks.append(test_prob)

    if not oof_blocks:
        raise ValueError(
            "No non-empty feature views were found."
        )

    # ---------------------------------------------------------
    # Stacking representation
    # ---------------------------------------------------------
    Ztr = np.column_stack(oof_blocks)
    Zte = np.column_stack(test_blocks)

    # Add simple probability interaction features
    # Helps meta-model learn agreement/disagreement across views.
    if len(oof_blocks) > 1:

        mean_tr = np.mean(
            np.stack(oof_blocks, axis=0),
            axis=0,
        )

        mean_te = np.mean(
            np.stack(test_blocks, axis=0),
            axis=0,
        )

        std_tr = np.std(
            np.stack(oof_blocks, axis=0),
            axis=0,
        )

        std_te = np.std(
            np.stack(test_blocks, axis=0),
            axis=0,
        )

        Ztr = np.column_stack([
            Ztr,
            mean_tr,
            std_tr,
        ])

        Zte = np.column_stack([
            Zte,
            mean_te,
            std_te,
        ])

    # ---------------------------------------------------------
    # Meta learner
    #
    # IMPORTANT:
    # multi_class='auto' REMOVED for newer sklearn.
    # lbfgs automatically handles multinomial classification.
    # ---------------------------------------------------------
    meta = LogisticRegression(
        C=0.35,
        class_weight="balanced",
        max_iter=5000,
        solver="lbfgs",
        random_state=seed,
    )

    meta.fit(
        Ztr,
        yi,
        sample_weight=balanced_weights(
            yi,
            power=0.55,
        ),
    )

    # ---------------------------------------------------------
    # Final validation prediction
    # ---------------------------------------------------------
    prob = meta.predict_proba(Zte)

    pp = np.argmax(prob, axis=1)

    pred = np.array(
        [VALID_LABELS[i] for i in pp],
        dtype=object,
    )

    gt = test_df["lirads_score"].astype(str).to_numpy()

    metrics = fast_challenge_score(
        gt,
        pred,
    )

    pred_df = pd.DataFrame({
        "case_id": test_df["case_id"].astype(str),
        "lirads_score": gt,
        "prediction": pred,
    })

    # Save probability columns too — useful for later ensembling
    for j, label in enumerate(VALID_LABELS):
        pred_df[f"p_{label}"] = prob[:, j]

    return {
        "metrics": metrics,
        "prediction": pred_df,
        "groups": used_groups,
        "meta_model": meta,
        "train_stack": Ztr,
        "test_stack": Zte,
    }


print("Patched multiview_stack_holdout_v2 loaded.")

Patched multiview_stack_holdout_v2 loaded.


In [63]:
mv=multiview_stack_holdout_v2(
    train_model,val_model,feature_cols,n_splits=N_SPLITS,seed=SEED,
    per_view_k=60 if FAST_MODE else 90
)
print(mv['metrics'])
print({k:len(v) for k,v in mv['groups'].items()})
mv['prediction'].to_csv(OUTPUT_DIR/'validation_predictions_multiview.csv',index=False)

f:\CORI_Final\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
f:\CORI_Final\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
f:\CORI_Final\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
f:\CORI_Final\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
f:\CORI_Final\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(


{'final_score': np.float64(0.7341353391050802), 'adjusted_qwk': np.float64(0.7411764714814302), 'special_category_recognition': 0.6942355889724311}
{'Morphology': 37, 'Custom kinetics': 90, 'Spatial fields': 76, 'Context + OT': 78, 'PyRadiomics': 90}


## 15. Oracle concept upper bound — analysis only

This deliberately uses radiologist-provided major-feature metadata at validation time. It is **not a deployable baseline**. Its purpose is to estimate how much headroom remains if concept recovery were perfect.

In [64]:
oracle=oracle_concept_holdout_v2(train_model,val_model,seed=SEED)
print('Oracle concept upper bound:',oracle['metrics'])

Oracle concept upper bound: {'final_score': np.float64(0.9351533225631539), 'adjusted_qwk': np.float64(0.9467084644042413), 'special_category_recognition': 0.869674185463659, 'accuracy': 0.847457627118644, 'macro_f1': 0.5384615384615384}


## 16. Primary paper comparison table

In [65]:
rows=[]
for _,r in baseline_table.iterrows(): rows.append(dict(r))
rows.append({'Method':'Simple hierarchical XGB',**add_common_metrics(val_model.lirads_score,simple_hier['prediction'].prediction)})
rows.append({'Method':'Cross-fitted multi-view stack',**mv['metrics']})
rows.append({'Method':'PHORA+ v2 (proposed)',**add_common_metrics(val_model.lirads_score,phora2['prediction'].prediction)})
rows.append({'Method':'Oracle concepts (upper bound; not deployable)',**oracle['metrics']})
primary=pd.DataFrame(rows).sort_values('final_score',ascending=False)
display(primary)
primary.to_csv(OUTPUT_DIR/'table_primary_validation.csv',index=False)

,Method,final_score,adjusted_qwk,special_category_recognition,accuracy,macro_f1
10,Oracle concepts (upper bound; not deployable),0.935153,9.467085e-01,0.869674,0.847458,0.538462
0,Extra Trees,0.762708,7.851852e-01,0.635338,0.644068,0.400319
8,Cross-fitted multi-view stack,0.734135,7.411765e-01,0.694236,NaN,NaN
9,PHORA+ v2 (proposed),0.717862,7.377327e-01,0.605263,0.610169,0.355204
1,Random Forest,0.705119,7.090301e-01,0.682957,0.644068,0.279381
7,Simple hierarchical XGB,0.692208,7.033493e-01,0.629073,0.593220,0.306170
2,Elastic-net logistic,0.689854,7.021277e-01,0.620301,0.576271,0.457360
3,Flat XGBoost,0.686617,6.857143e-01,0.691729,0.627119,0.282831
4,RBF-SVM,0.280690,2.322581e-01,0.555138,0.389831,0.228319
5,HistGradientBoosting,0.200213,1.134752e-01,0.691729,0.644068,0.279967


# Part III — Ablations (training split only)

These are the experiments to use for mechanism claims. Run them with `FAST_MODE=False` and the ablation switches enabled for the final manuscript.

## 17. Architecture ablations

- no concept/burden distillation;
- no factorized gate;
- no all-threshold ordinal expert;
- no clinical constraint router;
- no feature screening;
- no spatial burden distillation;
- full PHORA+ v2.

In [66]:
arch_rows=[]
if RUN_ARCH_ABLATIONS:
    specs=[
        ('PHORA+ v2 full',dict()),
        ('w/o concept + burden distillation',dict(use_concepts=False)),
        ('w/o factorized special gate',dict(use_factor_gate=False)),
        ('w/o all-threshold ordinal expert',dict(use_all_threshold=False)),
        ('w/o clinical constraints',dict(use_clinical_constraints=False)),
        ('w/o nested feature screening',dict(use_screen=False)),
    ]
    for i,(name,kw) in enumerate(specs):
        print('\n',name)
        rr=cross_validate_phora_plus_v2(
            train_model,feature_cols,n_splits=N_SPLITS,seed=SEED+i,
            screen_k=SCREEN_K,ensemble_seeds=ENSEMBLE_SEEDS,router_trials=ROUTER_TRIALS,
            use_concepts=kw.get('use_concepts',True),
            use_factor_gate=kw.get('use_factor_gate',True),
            use_all_threshold=kw.get('use_all_threshold',True),
            use_screen=kw.get('use_screen',True),
            use_clinical_constraints=kw.get('use_clinical_constraints',True)
        )
        arch_rows.append({'Method':name,**rr['metrics']})

    # Burden distillation ablation: retain binary concepts, remove only burden targets.
    no_burden=train_model.drop(columns=[c for c in train_model if c.startswith('burden_')],errors='ignore')
    rr=cross_validate_phora_plus_v2(
        no_burden,feature_cols,n_splits=N_SPLITS,seed=SEED+99,
        screen_k=SCREEN_K,ensemble_seeds=ENSEMBLE_SEEDS,router_trials=ROUTER_TRIALS
    )
    arch_rows.append({'Method':'w/o spatial burden distillation',**rr['metrics']})

arch_ablation=pd.DataFrame(arch_rows)
if len(arch_ablation):
    display(arch_ablation.sort_values('final_score',ascending=False))
    arch_ablation.to_csv(OUTPUT_DIR/'table_architecture_ablation_traincv.csv',index=False)
else:
    print('Set RUN_ARCH_ABLATIONS=True for the final ablation table.')


 PHORA+ v2 full
fold 1: selected=140 inner=0.4936 outer=0.4449
fold 2: selected=140 inner=0.6029 outer=0.4967
fold 3: selected=140 inner=0.5889 outer=0.5820

 w/o concept + burden distillation
fold 1: selected=140 inner=0.3800 outer=0.4385
fold 2: selected=140 inner=0.4316 outer=0.3954
fold 3: selected=140 inner=0.5570 outer=0.3096

 w/o factorized special gate
fold 1: selected=140 inner=0.5859 outer=0.5753
fold 2: selected=140 inner=0.4384 outer=0.5761
fold 3: selected=140 inner=0.5787 outer=0.2176

 w/o all-threshold ordinal expert
fold 1: selected=140 inner=0.6406 outer=0.5149
fold 2: selected=140 inner=0.5033 outer=0.6092
fold 3: selected=140 inner=0.6173 outer=0.3675

 w/o clinical constraints
fold 1: selected=140 inner=0.5615 outer=0.4920
fold 2: selected=140 inner=0.6766 outer=0.1064
fold 3: selected=140 inner=0.5539 outer=0.3186

 w/o nested feature screening
fold 1: selected=900 inner=0.4054 outer=0.5203
fold 2: selected=900 inner=0.4230 outer=0.3721
fold 3: selected=900 inne

,Method,final_score,adjusted_qwk,special_category_recognition
3,w/o all-threshold ordinal expert,0.508612,0.497367,0.572337
2,w/o factorized special gate,0.506814,0.503421,0.526041
0,PHORA+ v2 full,0.505430,0.496040,0.558639
6,w/o spatial burden distillation,0.409180,0.387014,0.534787
5,w/o nested feature screening,0.403101,0.379542,0.536604
1,w/o concept + burden distillation,0.391773,0.376455,0.478573
4,w/o clinical constraints,0.336266,0.293047,0.581174


## 18. Feature-family ablations

This answers whether the new spatial mathematics adds information beyond ordinary radiomics.

In [67]:
feature_rows=[]
if RUN_FEATURE_ABLATIONS:
    G=feature_groups_v2(feature_cols)
    specs={
        'Morphology only': G.get('Morphology',[]),
        'PyRadiomics only': G.get('PyRadiomics',[]),
        'Custom kinetics + spatial': list(dict.fromkeys(G.get('Custom kinetics',[])+G.get('Spatial fields',[]))),
        'Context + optimal transport only': G.get('Context + OT',[]),
        'All except PyRadiomics': [c for c in feature_cols if c not in set(G.get('PyRadiomics',[]))],
        'All inference-safe features': feature_cols,
    }
    for i,(name,cols) in enumerate(specs.items()):
        if len(cols)<3: continue
        print('\n',name,'n=',len(cols))
        rr=cross_validate_phora_plus_v2(
            train_model,cols,n_splits=N_SPLITS,seed=SEED+200+i,
            screen_k=min(SCREEN_K,len(cols)),ensemble_seeds=ENSEMBLE_SEEDS,
            router_trials=ROUTER_TRIALS
        )
        feature_rows.append({'Feature set':name,'n_features':len(cols),**rr['metrics']})
feature_ablation=pd.DataFrame(feature_rows)
if len(feature_ablation):
    display(feature_ablation.sort_values('final_score',ascending=False))
    feature_ablation.to_csv(OUTPUT_DIR/'table_feature_ablation_traincv.csv',index=False)
else:
    print('Set RUN_FEATURE_ABLATIONS=True for the final feature-family ablation table.')


 Morphology only n= 37
fold 1: selected=37 inner=0.3966 outer=0.6791
fold 2: selected=37 inner=0.6198 outer=0.5632
fold 3: selected=37 inner=0.6637 outer=0.0738

 PyRadiomics only n= 380
fold 1: selected=140 inner=0.4845 outer=0.7632
fold 2: selected=140 inner=0.6166 outer=0.4217
fold 3: selected=140 inner=0.7044 outer=0.2368

 Custom kinetics + spatial n= 396
fold 1: selected=140 inner=0.4490 outer=0.6055
fold 2: selected=140 inner=0.6071 outer=0.2181
fold 3: selected=140 inner=0.3304 outer=0.1915

 Context + optimal transport only n= 78
fold 1: selected=78 inner=0.5833 outer=0.4113
fold 2: selected=78 inner=0.4085 outer=0.6437
fold 3: selected=78 inner=0.3346 outer=0.2450

 All except PyRadiomics n= 520
fold 1: selected=140 inner=0.5824 outer=0.4737
fold 2: selected=140 inner=0.5512 outer=0.3939
fold 3: selected=140 inner=0.2908 outer=0.5501

 All inference-safe features n= 900
fold 1: selected=140 inner=0.4666 outer=0.5573
fold 2: selected=140 inner=0.6056 outer=0.3900
fold 3: sele

,Feature set,n_features,final_score,adjusted_qwk,special_category_recognition
1,PyRadiomics only,380,0.536475,0.533162,0.555249
5,All inference-safe features,900,0.492155,0.481632,0.551790
0,Morphology only,37,0.482392,0.473386,0.533426
3,Context + optimal transport only,78,0.476879,0.467802,0.528318
4,All except PyRadiomics,520,0.467123,0.458289,0.517183
2,Custom kinetics + spatial,396,0.335043,0.300160,0.532716


## 19. Repeatability / seed sensitivity of the full method

Run three repeated 3-fold CV experiments on the training split. Report mean ± SD in supplementary material.

In [68]:
repeat_rows=[]
if RUN_REPEATABILITY:
    for s in [42,142,242]:
        rr=cross_validate_phora_plus_v2(
            train_model,feature_cols,n_splits=N_SPLITS,seed=s,screen_k=SCREEN_K,
            ensemble_seeds=ENSEMBLE_SEEDS,router_trials=ROUTER_TRIALS
        )
        repeat_rows.append({'seed':s,**rr['metrics']})
repeatability=pd.DataFrame(repeat_rows)
if len(repeatability):
    display(repeatability)
    display(repeatability[['final_score','adjusted_qwk','special_category_recognition']].agg(['mean','std']))
    repeatability.to_csv(OUTPUT_DIR/'repeatability_traincv.csv',index=False)

fold 1: selected=140 inner=0.4936 outer=0.4449
fold 2: selected=140 inner=0.6029 outer=0.4967
fold 3: selected=140 inner=0.5889 outer=0.5820
fold 1: selected=140 inner=0.3944 outer=0.5863
fold 2: selected=140 inner=0.5041 outer=0.3685
fold 3: selected=140 inner=0.6709 outer=0.1807
fold 1: selected=140 inner=0.3423 outer=0.4484
fold 2: selected=140 inner=0.6014 outer=0.4291
fold 3: selected=140 inner=0.7211 outer=0.2714


,seed,final_score,adjusted_qwk,special_category_recognition
0,42,0.505430,0.496040,0.558639
1,142,0.440843,0.420005,0.558926
2,242,0.412100,0.386548,0.556894


,final_score,adjusted_qwk,special_category_recognition
mean,0.452791,0.434198,0.558153
std,0.047798,0.056109,0.001100


# Part IV — Validation analyses for the paper

## 20. Confusion matrix and per-class recall

In [69]:
vp=phora2['prediction']
gt=vp['lirads_score'].to_numpy(); pred=vp['prediction'].to_numpy()
cm=confusion_matrix(gt,pred,labels=VALID_LABELS)
cm_df=pd.DataFrame(cm,index=VALID_LABELS,columns=VALID_LABELS)
display(cm_df)
cm_df.to_csv(OUTPUT_DIR/'confusion_matrix_validation.csv')

rec=[]
for i,lab in enumerate(VALID_LABELS):
    denom=cm[i].sum(); rec.append({'class':lab,'n':int(denom),'recall':float(cm[i,i]/denom) if denom else np.nan})
per_class=pd.DataFrame(rec)
display(per_class)
per_class.to_csv(OUTPUT_DIR/'per_class_recall_validation.csv',index=False)

,LR-1,LR-2,LR-3,LR-4,LR-5,LR-M,LR-TIV
LR-1,0,1,0,0,0,0,0
LR-2,0,0,0,0,0,1,0
LR-3,0,0,1,0,1,1,0
LR-4,0,0,0,1,4,1,0
LR-5,0,0,0,1,22,4,0
LR-M,0,0,0,2,2,10,0
LR-TIV,0,0,0,1,1,3,2


,class,n,recall
0,LR-1,1,0.000000
1,LR-2,1,0.000000
2,LR-3,3,0.333333
3,LR-4,6,0.166667
4,LR-5,27,0.814815
5,LR-M,14,0.714286
6,LR-TIV,7,0.285714


## 21. Concept recovery on the untouched validation split

This measures whether the learned representation recovers the radiologists' major imaging concepts without receiving them as predictors.

In [70]:
concept_pairs={
    'Non-rim APHE':('aphe','latent_prob_aphe'),
    'Rim APHE':('rim_aphe','latent_prob_rim_aphe'),
    'Venous washout':('washout_venous','latent_prob_washout_venous'),
    'Delayed washout':('washout_delayed','latent_prob_washout_delayed'),
    'Venous capsule':('capsule_venous','latent_prob_capsule_venous'),
    'Delayed capsule':('capsule_delayed','latent_prob_capsule_delayed'),
}
concept_rows=[]
for name,(target,pcol) in concept_pairs.items():
    if pcol not in vp.columns: continue
    y=pd.to_numeric(val_model[target],errors='coerce').to_numpy(float)
    p=pd.to_numeric(vp[pcol],errors='coerce').to_numpy(float)
    ok=np.isfinite(y)&np.isfinite(p)
    if ok.sum()>=5 and len(np.unique(y[ok]))>1:
        concept_rows.append({'Concept':name,'n':int(ok.sum()),'AUROC':roc_auc_score(y[ok],p[ok]),'AUPRC':average_precision_score(y[ok],p[ok])})
concept_table=pd.DataFrame(concept_rows)
display(concept_table)
concept_table.to_csv(OUTPUT_DIR/'table_concept_recovery_validation.csv',index=False)

,Concept,n,AUROC,AUPRC
0,Non-rim APHE,52,0.915470,0.960180
1,Rim APHE,52,0.701587,0.245042
2,Venous washout,59,0.870238,0.787317
3,Delayed washout,59,0.905263,0.835260
4,Venous capsule,59,0.853968,0.730825
5,Delayed capsule,59,0.718182,0.164286


## 22. Spatial-burden distillation fidelity

For feature masks that exist in validation, correlate the predicted burden with the true annotated burden. This is an interpretability result, not a test-time input.

In [71]:
from scipy.stats import spearmanr, pearsonr
burden_rows=[]
for target in [c for c in val_model.columns if c.startswith('burden_')]:
    pcol='latent_'+target
    if pcol not in vp.columns: continue
    y=pd.to_numeric(val_model[target],errors='coerce').to_numpy(float)
    p=pd.to_numeric(vp[pcol],errors='coerce').to_numpy(float)
    ok=np.isfinite(y)&np.isfinite(p)
    if ok.sum()>=8:
        burden_rows.append({
            'target':target,'n':int(ok.sum()),
            'spearman_r':spearmanr(y[ok],p[ok]).statistic,
            'pearson_r':pearsonr(y[ok],p[ok]).statistic,
            'MAE':float(np.mean(np.abs(y[ok]-p[ok])))
        })
burden_fidelity=pd.DataFrame(burden_rows)
display(burden_fidelity)
burden_fidelity.to_csv(OUTPUT_DIR/'burden_distillation_validation.csv',index=False)

,target,n,spearman_r,pearson_r,MAE
0,burden_aphe,42,0.404262,0.582012,0.229024
1,burden_washout_venous,26,0.625343,0.516797,0.188095
2,burden_washout_delayed,20,0.330827,0.443762,0.234100
3,burden_capsule_venous,15,0.432143,0.553452,0.082435


## 23. Uncertainty via hierarchy disagreement

Two independent ordinal experts are available: regression and all-threshold. Their disagreement

\[
U_{ord}=|s_{reg}-s_{AT}|
\]

is combined with gate entropy to identify unstable cases. The challenge still receives one label; this is a failure-analysis / calibration result for the paper.

In [72]:
eps=1e-9
beta=float(phora2['router']['params'][0])
gd=vp[['p_direct_ord','p_direct_M','p_direct_TIV']].to_numpy(float)
gf=vp[['p_factor_ord','p_factor_M','p_factor_TIV']].to_numpy(float)
g=beta*gd+(1-beta)*gf; g=np.clip(g,eps,None); g/=g.sum(axis=1,keepdims=True)
vp_analysis=vp[['case_id','lirads_score','prediction','ord_reg_score','ord_AT_score']].copy()
vp_analysis['gate_entropy']=-(g*np.log(g)).sum(axis=1)/np.log(3)
vp_analysis['ordinal_disagreement']=np.abs(vp_analysis.ord_reg_score-vp_analysis.ord_AT_score)
vp_analysis['uncertainty']=vp_analysis['gate_entropy'] + .25*vp_analysis['ordinal_disagreement']
vp_analysis['correct']=(vp_analysis.lirads_score==vp_analysis.prediction).astype(int)
vp_analysis['uncertainty_quartile']=pd.qcut(vp_analysis.uncertainty,4,duplicates='drop')
uncertainty_table=vp_analysis.groupby('uncertainty_quartile',observed=True).agg(n=('case_id','size'),accuracy=('correct','mean'),mean_uncertainty=('uncertainty','mean')).reset_index()
display(uncertainty_table)
uncertainty_table.to_csv(OUTPUT_DIR/'uncertainty_validation.csv',index=False)

,uncertainty_quartile,n,accuracy,mean_uncertainty
0,"(0.081, 0.262]",15,0.800000,0.168470
1,"(0.262, 0.577]",15,0.533333,0.389332
2,"(0.577, 0.801]",14,0.642857,0.712858
3,"(0.801, 1.098]",15,0.466667,0.921658


## 24. Stratified bootstrap CIs + paired comparison against the best non-oracle baseline

In [73]:
def stratified_bootstrap_metrics(gt,pred,n_iter=5000,seed=42):
    gt=np.asarray(gt,object); pred=np.asarray(pred,object); rng=np.random.default_rng(seed)
    by={c:np.where(gt==c)[0] for c in np.unique(gt)}; rows=[]
    for _ in range(n_iter):
        idx=np.concatenate([rng.choice(ii,size=len(ii),replace=True) for ii in by.values()])
        rows.append(fast_challenge_score(gt[idx],pred[idx]))
    b=pd.DataFrame(rows)
    return pd.DataFrame([{'metric':c,'mean':b[c].mean(),'ci_low':b[c].quantile(.025),'ci_high':b[c].quantile(.975)} for c in b])

ci_phora=stratified_bootstrap_metrics(gt,pred,n_iter=BOOTSTRAP_ITERS,seed=SEED)
display(ci_phora)
ci_phora.to_csv(OUTPUT_DIR/'bootstrap_ci_phora_validation.csv',index=False)

# Choose the best deployable baseline by its already-computed holdout score.
best_name=baseline_table.iloc[0]['Method']
best_df=baseline_preds[best_name]
pair=vp[['case_id','lirads_score','prediction']].merge(best_df,on=['case_id','lirads_score'],suffixes=('_phora','_base'),validate='one_to_one')

def paired_bootstrap(pair,n_iter=5000,seed=42):
    y=pair.lirads_score.to_numpy(object); a=pair.prediction_base.to_numpy(object); b=pair.prediction_phora.to_numpy(object)
    rng=np.random.default_rng(seed); by={c:np.where(y==c)[0] for c in np.unique(y)}; ds=[]
    for _ in range(n_iter):
        idx=np.concatenate([rng.choice(ii,size=len(ii),replace=True) for ii in by.values()])
        ma=fast_challenge_score(y[idx],a[idx]); mb=fast_challenge_score(y[idx],b[idx])
        ds.append([mb[k]-ma[k] for k in ['final_score','adjusted_qwk','special_category_recognition']])
    z=np.asarray(ds)
    return pd.DataFrame({'metric':['final_score','adjusted_qwk','special_category_recognition'],
                         'mean_delta':z.mean(0),'ci_low':np.quantile(z,.025,axis=0),'ci_high':np.quantile(z,.975,axis=0),'P_delta_gt_0':(z>0).mean(0)})
paired=paired_bootstrap(pair,n_iter=BOOTSTRAP_ITERS,seed=SEED+1)
print('Comparator:',best_name)
display(paired)
paired.to_csv(OUTPUT_DIR/'paired_bootstrap_validation.csv',index=False)

,metric,mean,ci_low,ci_high
0,final_score,0.714659,0.551915,0.853218
1,adjusted_qwk,0.733963,0.541117,0.896553
2,special_category_recognition,0.605272,0.468672,0.750627


Comparator: Extra Trees


,metric,mean_delta,ci_low,ci_high,P_delta_gt_0
0,final_score,-0.052413,-0.129014,0.008053,0.0531
1,adjusted_qwk,-0.056271,-0.137631,0.008022,0.0523
2,special_category_recognition,-0.030553,-0.181704,0.119048,0.3473


## 25. Train→validation covariate shift: MMD + two-sample classifier

This is a quantitative domain-shift check. MMD is computed after training-only feature screening, median imputation, and standardization.

In [74]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression

sel=phora2['selected_features']
Xtr=train_model[sel].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).to_numpy(float)
Xva=val_model[sel].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).to_numpy(float)
imp=SimpleImputer(strategy='median').fit(Xtr); Xtr=imp.transform(Xtr); Xva=imp.transform(Xva)
sc=StandardScaler().fit(Xtr); Xtr=sc.transform(Xtr); Xva=sc.transform(Xva)

rng=np.random.default_rng(SEED); xs=Xtr[rng.choice(len(Xtr),min(250,len(Xtr)),replace=False)]; ys=Xva
D=pairwise_distances(np.vstack([xs,ys]),metric='euclidean'); nz=D[D>0]; sigma=np.median(nz) if len(nz) else 1.0; gamma=1/(2*sigma*sigma+1e-12)
def K(a,b): return np.exp(-gamma*pairwise_distances(a,b,metric='sqeuclidean'))
mmd2=float(K(xs,xs).mean()+K(ys,ys).mean()-2*K(xs,ys).mean())

Xd=np.vstack([Xtr,Xva]); yd=np.r_[np.zeros(len(Xtr)),np.ones(len(Xva))]
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
dp=cross_val_predict(LogisticRegression(max_iter=3000,class_weight='balanced'),Xd,yd,cv=cv,method='predict_proba')[:,1]
domain_auc=roc_auc_score(yd,dp)
shift=pd.DataFrame([{'MMD2_rbf':mmd2,'two_sample_domain_AUC':domain_auc,'n_features':len(sel)}])
display(shift)
shift.to_csv(OUTPUT_DIR/'train_validation_shift.csv',index=False)

,MMD2_rbf,two_sample_domain_AUC,n_features
0,0.007091,0.46112,140


## 26. Optional source/batch robustness

Because one batch is very small, treat leave-one-batch-out results as exploratory. The cell skips held-out batches with fewer than 10 lesion cases.

In [75]:
robust_rows=[]
if RUN_DOMAIN_ROBUSTNESS:
    pooled=pd.concat([train_model,val_model],ignore_index=True)
    for i,b in enumerate(sorted(pooled.batch_id.dropna().unique())):
        te=pooled[pooled.batch_id==b].reset_index(drop=True); tr=pooled[pooled.batch_id!=b].reset_index(drop=True)
        if len(te)<10: continue
        sel=nested_screen(tr,feature_cols,k=SCREEN_K,seed=SEED+i)
        Xtr=tr[sel].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).to_numpy(np.float32)
        Xte=te[sel].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).to_numpy(np.float32)
        enc={c:j for j,c in enumerate(VALID_LABELS)}; yi=np.array([enc[z] for z in tr.lirads_score],int)
        m=xgb_classifier(7,SEED+i,n_estimators=420,depth=3,lr=.025)
        m.fit(Xtr,yi,sample_weight=balanced_weights(yi,power=.65))
        pp=m.predict(Xte).astype(int); pr=np.array([VALID_LABELS[j] for j in pp],object)
        robust_rows.append({'held_out_batch':b,'n':len(te),**fast_challenge_score(te.lirads_score,pr)})
robust=pd.DataFrame(robust_rows)
if len(robust):
    display(robust)
    robust.to_csv(OUTPUT_DIR/'leave_one_batch_out_exploratory.csv',index=False)

,held_out_batch,n,final_score,adjusted_qwk,special_category_recognition
0,batch_001,43,0.236282,0.166667,0.630769
1,batch_002,223,0.677911,0.696306,0.573671
2,batch_003,249,0.326690,0.285347,0.560965


# Part V — Final freeze and challenge retraining

Only run this section **after** the method, feature set, and routing configuration are frozen from training CV + the single official validation analysis.

The saved object is a local research model (`joblib`) containing XGBoost estimators. It is useful for reproducing the frozen model and for building the final Codabench runtime. It is not by itself a Codabench ZIP.

In [76]:
RUN_FINAL_REFIT = False  # change to True only after method freeze

if RUN_FINAL_REFIT:
    import joblib
    all_public=pd.concat([train_model,val_model],ignore_index=True)
    final_model=fit_phora_plus_full_v2(
        all_public,feature_cols,seed=SEED,screen_k=SCREEN_K,
        ensemble_seeds=ENSEMBLE_SEEDS,router_trials=max(ROUTER_TRIALS,5000)
    )
    model_path=OUTPUT_DIR/'phora_plus_v2_all_public.joblib'
    joblib.dump(final_model,model_path,compress=3)
    manifest={
        'method':'PHORA+ v2','n_cases':int(len(all_public)),
        'class_counts':all_public.lirads_score.value_counts().to_dict(),
        'n_selected_features':len(final_model['selected']),
        'screen_k':SCREEN_K,'ensemble_seeds':list(ENSEMBLE_SEEDS),
        'router_trials':max(ROUTER_TRIALS,5000),
        'train_metadata':str(TRAIN_METADATA),'val_metadata':str(VAL_METADATA),
    }
    (OUTPUT_DIR/'final_training_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    print('Saved',model_path)
else:
    print('Final refit is OFF. Turn it on only after the method is frozen.')

Final refit is OFF. Turn it on only after the method is frozen.


## 28. Export manuscript-ready tables

In [77]:
# Save compact LaTeX tables where available.
def save_tex(df,name,floatfmt='%.3f'):
    if df is None or len(df)==0: return
    (OUTPUT_DIR/name).write_text(df.to_latex(index=False,float_format=lambda x: floatfmt % x),encoding='utf-8')

save_tex(primary,'table_primary_validation.tex')
if 'arch_ablation' in globals(): save_tex(arch_ablation,'table_architecture_ablation.tex')
if 'feature_ablation' in globals(): save_tex(feature_ablation,'table_feature_ablation.tex')
save_tex(concept_table,'table_concept_recovery.tex')
save_tex(per_class,'table_per_class_recall.tex')

print('Paper outputs:')
for p in sorted(OUTPUT_DIR.glob('*')):
    print(' ',p.name)

Paper outputs:
  annotation_burdens.csv
  bootstrap_ci_phora_validation.csv
  burden_distillation_validation.csv
  confusion_matrix_validation.csv
  context_ot_features.csv
  handcrafted_ibsi_radiomics.csv
  handcrafted_ibsi_radiomics_failures.csv
  leave_one_batch_out_exploratory.csv
  paired_bootstrap_validation.csv
  per_class_recall_validation.csv
  repeatability_traincv.csv
  spatial_physiology_features.csv
  table_architecture_ablation.tex
  table_architecture_ablation_traincv.csv
  table_baselines_validation.csv
  table_concept_recovery.tex
  table_concept_recovery_validation.csv
  table_feature_ablation.tex
  table_feature_ablation_traincv.csv
  table_per_class_recall.tex
  table_primary_validation.csv
  table_primary_validation.tex
  train_validation_shift.csv
  uncertainty_validation.csv
  validation_predictions_multiview.csv
  validation_predictions_phora_plus_v2.csv


# Recommended final experiment matrix for the paper

### Main table — official validation split
1. Majority class
2. Elastic-net logistic regression
3. RBF-SVM
4. Random Forest
5. Extra Trees
6. Histogram Gradient Boosting
7. Flat XGBoost
8. Simple hierarchical XGBoost
9. Cross-fitted multi-view stacking
10. **PHORA+ v2**
11. Oracle clinical concepts *(upper bound only; clearly marked non-deployable)*

### Architecture ablation — training CV
1. PHORA+ v2 full
2. − concept + burden distillation
3. − factorized special gate
4. − shared all-threshold ordinal expert
5. − clinically constrained router
6. − nested feature screening
7. − spatial burden distillation only

### Feature ablation — training CV
1. morphology only
2. conventional PyRadiomics only
3. custom kinetics + spatial fields
4. local context + optimal transport only
5. all except PyRadiomics
6. full multimodal radiomics

### Analyses
- concept AUROC/AUPRC;
- spatial-burden correlation;
- confusion matrix and class recall;
- stratified bootstrap 95% CI;
- paired bootstrap vs best baseline;
- ordinal-expert disagreement / uncertainty;
- MMD and train→validation two-sample AUC;
- optional source/batch robustness;
- repeated-seed CV.

## Final-run settings
Before producing numbers for the manuscript, set:

```python
FAST_MODE = False
RUN_ARCH_ABLATIONS = True
RUN_FEATURE_ABLATIONS = True
RUN_REPEATABILITY = True
```

Then restart the kernel and run top-to-bottom. Feature extraction is cached, so the expensive CT processing should not repeat.